In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# 1. Generar dataset simulado
fake = Faker()
np.random.seed(42)
n = 150

customer_id = [fake.uuid4() for _ in range(n)]
age = np.random.randint(18, 70, size=n)
gender = np.random.choice(['Male', 'Female'], size=n)
monthly_spent = np.round(np.random.uniform(20, 200, size=n), 2)
tenure_months = np.random.randint(1, 60, size=n)
num_support_calls = np.random.poisson(lam=2, size=n)

churn_prob = (
    0.6 * (monthly_spent < 80).astype(int) +
    0.3 * (tenure_months < 12).astype(int) +
    0.4 * (num_support_calls > 3).astype(int)
)
churn = np.where(churn_prob + np.random.rand(n) > 0.8, 1, 0)

df = pd.DataFrame({
    'customer_id': customer_id,
    'age': age,
    'gender': gender,
    'monthly_spent': monthly_spent,
    'tenure_months': tenure_months,
    'num_support_calls': num_support_calls,
    'churn': churn
})

# 2. Preparar datos
X = pd.get_dummies(df[['age','gender','monthly_spent','tenure_months','num_support_calls']], drop_first=True)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Entrenar modelo
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 4. Predicciones
y_pred = model.predict(X_test)

# 5. Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(cm)
print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")

# 6. Reporte de clasificación
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[22  4]
 [ 5 14]]
TP=14, FP=4, TN=22, FN=5

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.85      0.83        26
           1       0.78      0.74      0.76        19

    accuracy                           0.80        45
   macro avg       0.80      0.79      0.79        45
weighted avg       0.80      0.80      0.80        45

